# Exercise — Run Governance as an Operation

Governance is ongoing, not a one-time project. **Trailhead Provisions** gives you a single
audit that rolls up every governance signal — missing catalog metadata, quality failures,
lineage gaps, consent conflict, and **access-control gaps**. Your job: run it, **remediate what
is fixable in AWS (the catalog in Glue and access in Lake Formation), and re-run until the
platform is compliant** — then write up what remains as ongoing watch items. See `INSTRUCTIONS.md`.

> This exercise runs exclusively against **live AWS Glue, Lake Formation, and Amazon Athena** (no local fallback).

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog, S3 tables, and Lake Formation enforcement are seeded.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__}")
print(f"  Glue DB:          {gc.database}")
print(f"  Region:           {gc.region}")
print(f"  Athena workgroup: {gc.athena_workgroup}")
print(f"  Results bucket:   {gc.results_bucket}")

Backend: live -> AwsGovernedCatalog
  Glue DB:          trailhead
  Region:           us-east-1
  Athena workgroup: trailhead
  Results bucket:   s3://123456789012-trailhead-athena-results/


## 1. Run the platform governance audit (provided)

In [3]:
findings = gc.governance_audit(contracts_valid=True)
print("Total findings:", len(findings))
print("Initial access_control_gap objects:", findings[findings["signal"] == "access_control_gap"]["object"].tolist())
findings.groupby(["signal", "severity"]).size().reset_index(name="count")

Total findings: 24
Initial access_control_gap objects: ['customer (PII table, no Lake Formation restriction)']


,signal,severity,count
0,access_control_gap,high,1
1,consent_conflict,high,1
2,missing_lineage,medium,4
3,missing_metadata,high,3
4,missing_metadata,medium,7
5,quality_violation,high,5
6,quality_violation,low,3


## 2. Remediate the catalog gaps in Glue
Close every `missing_metadata` finding by writing domain-specific `owner`, `classification`, and `retention` metadata (`gc.set_table_metadata`) and tagging untagged columns (`gc.tag_column`) as `PII`, `SENSITIVE`, or `PUBLIC`.

In [4]:
TABLE_REMEDIATION = {
    "orders":              {"owner": "orders-team@trailhead.example"},
    "supplier":            {"retention": "P5Y"},
    "product":             {"retention": "P5Y"},
    "shipment":            {"classification": "PUBLIC"},
    "campaign":            {"owner": "marketing@trailhead.example"},
    "campaign_membership": {"owner": "marketing@trailhead.example", "classification": "PII", "retention": "P3Y"},
}
for tbl, meta in TABLE_REMEDIATION.items():
    gc.set_table_metadata(tbl, **meta)

PII_COLS = {"email", "phone", "cust_ref", "full_name"}
SENSITIVE_COLS = {"unit_cost", "amount", "marketing_consent", "mkt_consent"}
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    for col in tm.columns:
        if col.classification is None:
            cls = "PII" if col.name in PII_COLS else ("SENSITIVE" if col.name in SENSITIVE_COLS else "PUBLIC")
            gc.tag_column(tbl, col.name, cls)

mid_findings = gc.governance_audit(contracts_valid=True)
print("Catalog gaps remaining:", len(gc.catalog_audit()))
print("access_control_gap objects after PII tagging:", mid_findings[mid_findings["signal"] == "access_control_gap"]["object"].tolist())

Catalog gaps remaining: 0
access_control_gap objects after PII tagging: ['customer (PII table, no Lake Formation restriction)', 'campaign_membership (PII table, no Lake Formation restriction)']


## 3. Remediate the access-control gaps in Lake Formation
Every **PII table** (`customer` and `campaign_membership`) must be governed by Lake Formation. Bring each PII-bearing table under an LF-Tag (`governed=true`) and grant the steward (`co_data_steward`) by tag.

In [5]:
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    if tm.classification == "PII" or any(c.classification == "PII" for c in tm.columns):
        gc.assign_tag(tbl, "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")
print("Access governed for the PII tables.")
display(gc.grants_summary())

Access governed for the PII tables.


,role,table,access
0,co_marketing_analyst,(none),no grants
1,co_cs_rep,(none),no grants
2,co_data_steward,campaign_membership,via LF-Tag (all cols/rows)
3,co_data_steward,customer,via LF-Tag (all cols/rows)
4,co_auditor,(none),no grants


## 4. Re-audit and confirm compliance (provided)
The AWS-fixable findings — `missing_metadata` and `access_control_gap` — should both be zero. What remains are the recurring SLOs (quality, consent, lineage).

In [6]:
re_findings = gc.governance_audit(contracts_valid=True)
def n(df, sig): return int((df["signal"] == sig).sum())
print("missing_metadata: ", n(findings, "missing_metadata"),  "->", n(mid_findings, "missing_metadata"),  "->", n(re_findings, "missing_metadata"))
print("access_control_gap:", n(findings, "access_control_gap"), "->", n(mid_findings, "access_control_gap"), "->", n(re_findings, "access_control_gap"))
compliant = n(re_findings, "missing_metadata") == 0 and n(re_findings, "access_control_gap") == 0
print("AWS-fixable findings cleared?", compliant)
re_findings.groupby(["signal", "severity"]).size().reset_index(name="count")

missing_metadata:  10 -> 0 -> 0
access_control_gap: 1 -> 2 -> 0
AWS-fixable findings cleared? True


,signal,severity,count
0,consent_conflict,high,1
1,missing_lineage,medium,4
2,quality_violation,high,5
3,quality_violation,low,3


## 5. Operations write-up
Replace the cell below with your write-up. Address every requirement in `INSTRUCTIONS.md`:
- What you remediated in Glue (catalog) and in Lake Formation (access), and the before/after counts (`missing_metadata: 10 -> 0`, `access_control_gap: 1 -> 2 -> 0`).
- Why classifying untagged PII columns in Step 2 surfaced a new access-control gap on `campaign_membership` (while `customer` was already flagged in Step 1).
- **One concrete SLO target per remaining signal category** (`quality_violation`, `consent_conflict`, `missing_lineage`), clearly separating **current instance remediation** from **recurring operational SLO monitoring**.
- How you'd operate this audit continuously (schedule, ownership, trending).

### Governance operations write-up

**1. Control-plane remediation in AWS Glue and AWS Lake Formation (before/after counts):**
- **Catalog (`missing_metadata`: `10 -> 0 -> 0`):** In Step 2, `AwsGovernedCatalog` updated live AWS Glue table parameters and column classifications across all 6 tables with gaps. Rather than assigning tables to central governance or blanket-tagging every non-PII column as `PUBLIC`, each table was assigned to its domain owner (`orders` -> `orders-team@trailhead.example`, `campaign` and `campaign_membership` -> `marketing@trailhead.example`), appropriate retention (`P5Y` on `supplier` and `product`, `P3Y` on `campaign_membership`), and accurate column sensitivity (`campaign_membership.cust_ref` as `PII`, `product.unit_cost` as `SENSITIVE`, and `shipment.carrier` / `campaign.channel` as `PUBLIC`). Re-running `gc.catalog_audit()` confirmed `0` catalog gaps remaining.
- **Access (`access_control_gap`: `1 -> 2 -> 0`):** In Step 1's initial audit, `customer` was **already** flagged as an `access_control_gap` (`1` finding: `['customer (PII table, no Lake Formation restriction)']`) because `customer` was already classified as `PII` in Glue without Lake Formation tag governance. When Step 2 classified `campaign_membership.cust_ref` as `PII` (and `campaign_membership` as a `PII` table), `campaign_membership` became a second PII-bearing table, causing `access_control_gap` to rise from `1` to `2` (`['customer (PII table, no Lake Formation restriction)', 'campaign_membership (PII table, no Lake Formation restriction)']`). In Step 3, we assigned the `governed=true` LF-Tag to both `customer` and `campaign_membership` and granted `co_data_steward` access via `grant_by_tag("co_data_steward", "governed", "true")`, driving `access_control_gap` from `2` to `0`.

**2. Remaining signals — Concrete SLO targets and Current Instance Remediation vs. Recurring SLO Monitoring:**
After clearing the 11 control-plane findings (`10` catalog + `1` initial access gap), **11 findings** remain across three recurring operational signal categories. For each category, we separate **current instance remediation** (fixing today's broken table/pipeline) from **recurring operational SLO monitoring** (watching the metric on every scheduled audit run because new tables, schema changes, and data loads continuously re-introduce drift):

- **`quality_violation` (8 findings: 6 high, 2 low)**:
  - **Concrete SLO targets**: completeness `>= 99.0%` non-null (`product.category`, `product.sustainability_class`), validity `>= 99.5%` valid format (`customer.email`, `customer.phone`), accuracy `>= 99.9%` non-negative (`orders.amount`) and `100%` in kg (`0 < carbon_kg <= 100` on `shipment.carbon_kg`), timeliness `>= 98.0%` within 1,095 days (`customer.created_ts`).
  - **Current instance remediation**: Ticket the domain stewards (`inventory-fulfillment@`, `customer-identity@`, `orders-team@`) to clean the current defective batches (e.g., normalizing the grams-vs-kg records in `shipment.carbon_kg` and quarantining invalid `n/a` emails).
  - **Recurring operational SLO monitoring**: Run `run_quality_suite()` on every scheduled ingestion batch and weekly audit, because new vendor shipments, checkout bugs, and partner feeds continuously arrive and can violate completeness, validity, accuracy, or timeliness thresholds at any time.

- **`consent_conflict` (1 finding: high)**:
  - **Concrete SLO target**: cross-domain consent mismatch rate `< 1.0%` (`>= 99.0%` consistency between `campaign_membership.mkt_consent` and the loyalty system of record `customer.marketing_consent`).
  - **Current instance remediation**: Reconcile the current ~19.8% opt-in disagreement by running a backfill sync from the loyalty system of record (`customer.marketing_consent`) into marketing's `campaign_membership.mkt_consent`.
  - **Recurring operational SLO monitoring**: Monitor the cross-domain consent consistency check on every weekly audit and campaign load, because new marketing campaign imports and regional sign-up forms can re-introduce un-synced consent flags if an ingestion path bypasses the loyalty SoR.

- **`missing_lineage` (2 findings: medium)**:
  - **Concrete SLO target**: `100%` of production Glue catalog tables covered by OpenLineage job events (`0` un-instrumented tables).
  - **Current instance remediation**: Ticket Platform Engineering to add OpenLineage emitters to the ETL jobs feeding the two currently un-instrumented tables (`order_returns` and `campaign`).
  - **Recurring operational SLO monitoring**: Check lineage coverage on every weekly `governance_audit()` run (and enforce it in the onboarding checklist), because new tables and ad-hoc pipelines are added every quarter and would otherwise create blind spots for breach/impact analysis.

**3. Continuous operating model:**
Schedule `governance_audit()` weekly via EventBridge + Step Functions against live AWS Glue, Lake Formation, and Athena. Any `missing_metadata` or `access_control_gap` regression pages the Central Governance Lead and Security immediately (`0` tolerance), while `quality_violation`, `consent_conflict`, and `missing_lineage` metrics are trended weekly on the Governance Posture Dashboard against their SLO targets and routed to the owning domain teams.

---
## 6. Governance Dashboard Specification

Design a dashboard that a governance team would check weekly. It should surface the audit
findings you just triaged, track trends over time, and make it obvious when the platform
is drifting from compliance.

**Include:**
- A top-line KPI (single governance health number)
- At least 3 panels/views with specific metrics, thresholds, and owners
- Alerting rules (what triggers a notification and to whom)

### Dashboard spec — Trailhead Governance Posture

**Top-line KPI (single number)**:
- `Governance Score` = 100 × (1 − high-severity findings / total audited objects)
- Target: ≥ 95%. Alert at < 90%.

**Panel 1: Findings & SLOs by signal (bar chart + threshold table, stacked by severity)**

| Signal | Metric | Concrete SLO / Threshold | Owner |
|--------|--------|--------------------------|-------|
| `missing_metadata` | Count of Glue catalog gaps | `0` open findings (100% metadata completeness) | Central Data Governance |
| `access_control_gap` | Count of ungoverned PII tables in LF | `0` open findings (100% PII tables LF-governed) | Data Security / Governance |
| `quality_violation` | Dimension pass rate (`completeness`, `validity`, `accuracy`, `timeliness`) | Completeness `>= 99.0%`, Validity `>= 99.5%`, Accuracy `>= 99.9%` (`100%` on `carbon_kg <= 100`), Timeliness `>= 98.0%` | Domain Data Stewards |
| `consent_conflict` | Cross-domain consent mismatch rate (`mkt_consent` vs `marketing_consent`) | `< 1.0%` mismatch (`>= 99.0%` consistency with loyalty SoR) | Customer Identity & Marketing Stewards |
| `missing_lineage` | % of Glue catalog tables covered by OpenLineage events | `100%` table coverage (`0` un-instrumented tables) | Platform Engineering |

**Panel 2: Trend over time (line chart, weekly cadence)**
- X-axis: weekly `governance_audit()` run date
- Y-axis: finding count and SLO pass rate per signal
- Tracks whether current-instance fixes hold or whether new tables/loads introduce drift

**Panel 3: Domain health heatmap**
- Rows: 4 domains (`Customer Identity & Loyalty`, `E-commerce Orders`, `Inventory & Fulfillment`, `Marketing Campaigns`)
- Columns: Catalog Complete, LF Access Governed, Quality SLOs Met, Lineage Covered, Consent Aligned
- Cells: Green (meets SLO), Yellow (low-severity warning), Red (SLO breached)

**Alerting rules:**
- Any finding (`> 0`) on `missing_metadata` or `access_control_gap` → PagerDuty alert to Central Governance / Security immediately
- `quality_violation` breaching dimension SLO (or increasing 2 consecutive weeks) → Slack alert + Jira ticket to owning domain steward
- `consent_conflict` mismatch rate `>= 1.0%` → escalate to DPO + Customer Identity & Marketing leads
- `missing_lineage` `> 0` un-instrumented tables → automated ticket to Platform Engineering

---
## 7. New-Domain Onboarding Checklist

A 5th domain team is joining the platform next quarter. Write the checklist they must
complete before their first table goes live — so the governance audit stays green.

**Include:**
- At least 6 steps covering catalog, PII, retention, access, lineage, and quality
- For each step: what to do, and what governance signal it satisfies
- A verification step (how the new team confirms they're compliant)

### New-domain onboarding checklist

Before your first table goes live on the Trailhead platform, complete every item below.
The governance audit runs weekly against live AWS Glue, Lake Formation, and Athena — any unchecked item will surface as a finding.

| # | Step | What to do | Governance signal satisfied |
|---|------|-----------|-----------------------------|
| 1 | **Register table metadata in Glue** | Set domain `owner` (e.g., `new-domain@trailhead.example`), table `classification`, and ISO-8601 `retention` in Glue table parameters | `missing_metadata == 0` (table-level) |
| 2 | **Classify every column in Glue** | Tag every column's `classification` parameter in Glue as `PII` (personal data), `SENSITIVE` (commercial/financial/consent), or `PUBLIC` | `missing_metadata == 0` (`untagged_column == 0`) |
| 3 | **Declare retention schedule** | Configure ISO-8601 `retention` (e.g., `P3Y` for PII, `P5Y` for transactional/commercial) and register with lifecycle purge job | Retention compliance & contract `retention` check |
| 4 | **Govern access in Lake Formation** | Revoke `IAMAllowedPrincipals`, assign `governed=true` LF-Tag to any PII-bearing table, grant `co_data_steward` by tag, and configure least-privilege column/row filters for consumers | `access_control_gap == 0` |
| 5 | **Instrument OpenLineage events** | Configure ingestion/transform jobs to emit OpenLineage `inputs` and `outputs` edges for every new table | `missing_lineage == 0` (`100%` table coverage SLO) |
| 6 | **Pass data quality SLO baseline** | Add Great Expectations checks and verify completeness `>= 99.0%`, validity `>= 99.5%`, accuracy `>= 99.9%`, timeliness `>= 98.0%` via `run_quality_suite()` | `quality_violation == 0` on domain tables |
| 7 | **Publish & validate data contract** | Commit JSON Schema contract with `pii_tags` matching `col['pii']=True`, `retention`, and `quality_slos` (`0 < threshold <= 1`); verify `validate_contract()` returns `True` | `contract_invalid == 0` |
| 8 | **Bind to consent SoR (if applicable)** | If processing customer communications, join against loyalty's `customer.marketing_consent` system of record rather than maintaining a separate consent flag | `consent_conflict` mismatch `< 1.0%` SLO |

**Verification step**: Run `gc.governance_audit(contracts_valid=True)` and `gc.grants_summary()` in staging before cutover. Confirm `missing_metadata == 0`, `access_control_gap == 0`, `missing_lineage == 0` for the new domain's tables, and all quality SLO expectations pass.

---
## 8. Cross-Module Synthesis — Mini Architect's Brief

You've now touched every governance discipline across MP1–MP9. The capstone project will ask
you to write a full **Architect's Brief** — a 1–2 page executive synthesis of governance posture,
tradeoffs, and a roadmap.

**Practice that skill now.** Write a short (3–5 paragraph) mini-brief for Trailhead Provisions
that connects your findings from at least 3 different modules. This is NOT a summary of each
module — it's a synthesis showing how findings in one area drove decisions in another.

**Structure:**
1. **Posture statement** — In one sentence, where does the platform stand right now?
2. **Key tradeoff** — Name one tension you navigated (e.g., restricting access vs. enabling analytics) and how you resolved it.
3. **Cross-module connection** — Show how a finding from an earlier module (MP2–MP8) directly informed a decision in a later module. Be specific.
4. **One-quarter roadmap** — 3 actions for next quarter, referencing evidence from your exercises.

### Mini Architect's Brief — Trailhead Provisions Governance Posture

**Posture**: Trailhead Provisions now enforces live AWS Glue catalog completeness (`missing_metadata: 10 -> 0`) and AWS Lake Formation tag-based and column/row access controls across all PII-bearing tables (`access_control_gap: 1 -> 2 -> 0`), while managing data quality, cross-domain consent consistency, and OpenLineage coverage under explicit operational SLOs.

**Key tradeoff — access restriction vs. analytics enablement**: Remediating untagged PII columns in the Glue catalog (MP2 / MP9 Step 2 — tagging `campaign_membership.cust_ref` as `PII` alongside `customer`'s `full_name`, `email`, and `phone`) immediately expanded our Lake Formation compliance scope from one PII table (`customer`) to two (`customer` and `campaign_membership`). Locking down PII tables entirely would have blocked marketing segmentation and EU customer support. We resolved this tension in Lake Formation (MP7 / MP9 Step 3) using fine-grained controls: `co_marketing_analyst` receives a column-level grant excluding `full_name`, `email`, and `phone` while retaining segmentation attributes (`loyalty_tier`, `region`), `co_cs_rep` receives a row-filtered DataCellsFilter (`region = 'EU'`), and `co_data_steward` receives scalable LF-TBAC access across all tables tagged `governed=true`.

**Cross-module connection**: The ~19.8% consent mismatch first discovered during MP2's catalog and semantic-conflict audit (`customer.marketing_consent` at ~60% opt-in vs. `campaign_membership.mkt_consent` at ~80% opt-in) directly drove our MP4 consistency quality check, revealed its blast radius in MP6 when OpenLineage traced `customer` and `campaign_membership` through `customer_360` into `marketing_audience`, and surfaced in MP9 as a high-severity `consent_conflict` operational risk. Seeing the same root cause across catalog, quality, lineage, and operations proved that ad-hoc SQL cleanups are insufficient — which is why our Section 6 dashboard enforces a `< 1.0%` consent mismatch SLO and our Section 7 onboarding checklist requires any new domain to bind directly to the loyalty consent system of record.

**One-quarter roadmap**:
1. **Enforce loyalty as the single Consent System of Record (closes `consent_conflict` to `< 1.0%` SLO)** — Replace `campaign_membership.mkt_consent` writes with an automated sync from `customer.marketing_consent` and quarantine unresolvable `n/a` customer references before `build_customer_360` runs.
2. **Quarantine unit-of-measure and format defects at ingestion (closes `quality_violation` to meet `100%` / `>= 99.5%` SLOs)** — Enforce data contract gates on `shipment.carbon_kg <= 100` (fixing the grams-vs-kg defect that inflated CSRD Scope 3 disclosures in MP8) and regex validation on `customer.email` / `phone`.
3. **Complete OpenLineage instrumentation & automate weekly `governance_audit()` runs (closes `missing_lineage` to `100%` coverage)** — Instrument `order_returns` and `campaign` ETL jobs in Platform Engineering and run `governance_audit()` weekly via Step Functions, paging on any `missing_metadata` or `access_control_gap` regression.

---
# Solution — Run Governance as an Operation

## Summary

This exercise practices **operating** governance against **live AWS Glue, Lake Formation, and Amazon
Athena** (no local fallback): running a platform-wide audit (`AwsGovernedCatalog` reads live Glue
table/column classifications and live Lake Formation tags/permissions), driving the AWS-fixable
control-plane findings to zero (catalog in Glue, access in Lake Formation), and managing the
remaining signals (`quality_violation`, `consent_conflict`, `missing_lineage`) with both
current-instance remediation and recurring operational SLO monitoring.

## Key steps

Close catalog gaps using domain owners and proper sensitivity classifications, then close Lake Formation access gaps on all PII tables, then re-audit:

```python
# Step 2 — Glue: domain-specific table metadata + column classifications (PII / SENSITIVE / PUBLIC)
TABLE_REMEDIATION = {
    "orders":              {"owner": "orders-team@trailhead.example"},
    "supplier":            {"retention": "P5Y"},
    "product":             {"retention": "P5Y"},
    "shipment":            {"classification": "PUBLIC"},
    "campaign":            {"owner": "marketing@trailhead.example"},
    "campaign_membership": {"owner": "marketing@trailhead.example", "classification": "PII", "retention": "P3Y"},
}
for tbl, meta in TABLE_REMEDIATION.items():
    gc.set_table_metadata(tbl, **meta)

PII_COLS = {"email", "phone", "cust_ref", "full_name"}
SENSITIVE_COLS = {"unit_cost", "amount", "marketing_consent", "mkt_consent"}
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    for col in tm.columns:
        if col.classification is None:
            cls = "PII" if col.name in PII_COLS else ("SENSITIVE" if col.name in SENSITIVE_COLS else "PUBLIC")
            gc.tag_column(tbl, col.name, cls)

mid_findings = gc.governance_audit(contracts_valid=True)
# missing_metadata: 10 -> 0
# access_control_gap: 1 -> 2 (customer was already 1; tagging campaign_membership.cust_ref as PII surfaced a 2nd!)

# Step 3 — Lake Formation: every PII-bearing table (customer, campaign_membership) must be governed
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    if tm.classification == "PII" or any(c.classification == "PII" for c in tm.columns):
        gc.assign_tag(tbl, "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")
display(gc.grants_summary())

# Step 4 — Re-audit
re_findings = gc.governance_audit(contracts_valid=True)   # missing_metadata == 0, access_control_gap == 0
```

## Remaining Signals & Concrete SLO Targets

After clearing `missing_metadata` (`10 -> 0`) and `access_control_gap` (`1 -> 2 -> 0`), 11 findings
remain across three recurring operational signals. For each category, separate **current instance
remediation** from **recurring operational SLO monitoring**:

- **`quality_violation` (8 findings — 6 high, 2 low)**:
  - *Concrete SLO targets*: completeness `>= 99.0%` non-null (`product.category`, `product.sustainability_class`),
    validity `>= 99.5%` valid format (`customer.email`, `customer.phone`), accuracy `>= 99.9%` non-negative
    (`orders.amount`) and `100%` in kg (`0 < carbon_kg <= 100` on `shipment.carbon_kg`), timeliness `>= 98.0%`
    within 1,095 days (`customer.created_ts`).
  - *Current instance remediation vs. recurring SLO*: Ticket domain stewards to clean current defective batches
    (e.g., normalizing the grams slice in `shipment.carbon_kg`), and monitor quality SLOs on every ingestion
    load because upstream vendor feeds and schema changes continuously introduce new defects.
- **`consent_conflict` (1 finding — high)**:
  - *Concrete SLO target*: cross-domain consent mismatch rate `< 1.0%` (`>= 99.0%` consistency between
    `campaign_membership.mkt_consent` and the loyalty system of record `customer.marketing_consent`).
  - *Current instance remediation vs. recurring SLO*: Reconcile the current ~20% opt-in disagreement by
    syncing marketing's table from loyalty's authoritative consent store, and monitor the `< 1.0%` mismatch
    SLO continuously because new campaign imports and partner lists can re-introduce un-synced consent flags.
- **`missing_lineage` (2 findings — medium)**:
  - *Concrete SLO target*: `100%` of production Glue catalog tables covered by OpenLineage job events
    (`0` un-instrumented tables).
  - *Current instance remediation vs. recurring SLO*: Ticket Platform Engineering to emit OpenLineage events
    for the two currently un-instrumented tables (`order_returns` and `campaign`), and monitor the `100%`
    lineage coverage SLO on every weekly audit because new domain tables and pipelines are onboarded every quarter.

## Common Mistakes to Avoid

- **Tagging every non-PII column as `PUBLIC`** — commercially sensitive columns like `product.unit_cost` must be tagged `SENSITIVE`, and domain tables must be assigned to their domain owners (`orders-team@`, `marketing@`), not central governance.
- **Claiming `customer` had no access-control gap before Step 2** — `customer` was already flagged as an `access_control_gap` in the initial Step 1 audit (`1` finding); classifying `campaign_membership.cust_ref` as `PII` in Step 2 added `campaign_membership` as a second finding (`1 -> 2`), before Step 3 governed both (`2 -> 0`).
- **Trying to "fix" quality/consent/lineage in the catalog or conflating one-time tickets with SLOs** — separate fixing today's instances (e.g., instrumenting `order_returns` and `campaign`) from monitoring the recurring SLOs on every scheduled audit run.
- **Closing catalog gaps but not the access gaps they create** — classifying PII columns makes their tables subject to access governance; you must then govern them in Lake Formation and verify with `gc.grants_summary()` and `gc.governance_audit()`.